# Does fine-tuning memorize subjects, and where does a leaky network look?

Two checks the reviewers asked for, on fold 1 of the runs in the paper (same data, folds, seeds and training recipe as the controlled cross validation run).

1. **Training fit under the subject split.** The three One-vs-All hybrid networks and the four frozen-feature logistic regressions are scored on
   their own fitting subjects and on the test subjects. If the fine-tuned network fits its training subjects almost perfectly while the frozen-feature
   classifiers do not, that supports the memorization reading of Table I.
2. **Grad-CAM, slice split versus subject split.** The AD-vs-rest hybrid is trained once under the subject split and once under the random slice split,
   and both are explained on the same slices (test slices in both splits). For each slice the notebook measures how much Grad-CAM mass falls outside the
   brain, on the outer 12-pixel rim of the brain and in its centre , next to the share of the image each region covers.

In [ ]:
import os, json, math, random, shutil, time, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from PIL import Image
from scipy import stats, ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (resnet18, ResNet18_Weights, densenet121, DenseNet121_Weights,
                                resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

Device: cuda


In [ ]:
INPUT_ROOT = '/kaggle/input'
WORK_DIR   = '/kaggle/working'
OUT_DIR = os.path.join(WORK_DIR, 'results_memorization_xai')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

IMAGE_SIZE  = 224
BATCH_SIZE  = 32
NUM_WORKERS = 2
N_CAM  = 300
N_BOOT = 2000

SEED = 42
N_OUTER      = 5
CAL_FRACTION = 0.20
PRETRAINED   = True
VALID_EXTS   = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLASS_NAMES = ['AD', 'CN', 'MCI']
AD, CN, MCI = 0, 1, 2
BACKBONES = ['resnet18+densenet121', 'resnet50', 'densenet121', 'efficientnet_b0']
PCA_DIM = 256
RANDAUG_N, RANDAUG_M = 2, 7
RANDOM_ERASING  = 0.20

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)

# hybrid recipe
PHASE_A_EPOCHS  = 8
PHASE_B_EPOCHS  = 15
WARMUP_EPOCHS   = 2
LR_MIN          = 1e-6
LR_HEAD_A       = 3e-3
LR_HEAD_B       = 1e-3
LR_BACKBONE_B   = 1e-4
WEIGHT_DECAY    = 5e-5
GRAD_CLIP_NORM  = 1.0
HEAD_HIDDEN     = 384
DROPOUT         = 0.30
LABEL_SMOOTHING = 0.05
OVA_TASKS = ['AD_vs_REST', 'CN_vs_REST', 'MCI_vs_REST']

In [ ]:
DATA_ROOT = None
for root, dirs, _ in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'adni_nifti_preprocessed' and {'AD', 'CN', 'MCI'} <= set(dirs):
        DATA_ROOT = root
        break
assert DATA_ROOT, 'Attach the ADNI dataset (a folder adni_nifti_preprocessed with AD, CN and MCI inside)'

patient_classes = defaultdict(set)
patient_files = defaultdict(list)
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_ROOT, cls)
    for pid in sorted(os.listdir(cls_dir)):
        pdir = os.path.join(cls_dir, pid)
        if not os.path.isdir(pdir):
            continue
        files = sorted(os.path.join(pdir, f) for f in os.listdir(pdir) if os.path.splitext(f)[1].lower() in VALID_EXTS)
        if files:
            patient_classes[pid].add(cls)
            patient_files[(pid, cls)] = files

patients = sorted(p for p, c in patient_classes.items() if len(c) == 1)
patient_label = {p: CLASS_NAMES.index(next(iter(patient_classes[p]))) for p in patients}

slice_path, slice_pid, slice_label = [], [], []
for p in patients:
    for f in patient_files[(p, CLASS_NAMES[patient_label[p]])]:
        slice_path.append(f)
        slice_pid.append(p)
        slice_label.append(patient_label[p])
slice_pid = np.array(slice_pid)
slice_label = np.array(slice_label)
n_slices = len(slice_path)

pid_idx = defaultdict(list)
for i, p in enumerate(slice_pid):
    pid_idx[p].append(i)
pid_idx = {p: np.array(v) for p, v in pid_idx.items()}

def idx_of(pids):
    return np.concatenate([pid_idx[p] for p in pids])

IMAGES = np.zeros((n_slices, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
for i, f in enumerate(slice_path):
    IMAGES[i] = np.array(Image.open(f).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS))

normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
hflip = transforms.RandomHorizontalFlip(p=0.5)
rand_aug = transforms.RandAugment(num_ops=RANDAUG_N, magnitude=RANDAUG_M)
jitter = transforms.ColorJitter(brightness=0.15, contrast=0.15)
erasing = transforms.RandomErasing(p=RANDOM_ERASING, scale=(0.02, 0.12))

class SliceSet(Dataset):
    def __init__(self, idx, labels, augment):
        self.idx = np.asarray(idx)
        self.labels = np.asarray(labels)
        self.augment = augment

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        t = torch.from_numpy(IMAGES[self.idx[i]].copy()).unsqueeze(0).repeat(3, 1, 1)
        if self.augment:
            t = rand_aug(hflip(t))
            t = erasing(normalize(jitter(t.float() / 255.0)))
        else:
            t = normalize(t.float() / 255.0)
        return t, int(self.labels[i])

def make_loader(idx, labels, augment, shuffle):
    return DataLoader(SliceSet(idx, labels, augment), batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

counts = {c: sum(1 for p in patients if patient_label[p] == i) for i, c in enumerate(CLASS_NAMES)}
print(f'Subjects: {len(patients)} {counts} | slices: {n_slices}')

Subjects: 320 {'AD': 54, 'CN': 88, 'MCI': 178} | slices: 8512


In [ ]:
def make_head(in_dim, num_classes):
    return nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_dim, HEAD_HIDDEN), nn.BatchNorm1d(HEAD_HIDDEN), nn.GELU(),
                         nn.Dropout(DROPOUT * 0.5), nn.Linear(HEAD_HIDDEN, num_classes))

class ResNetDenseNetHybrid(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        self.resnet, self.densenet = r, d
        self.head = make_head(512 + 1024, num_classes)

    def forward(self, x):
        return self.head(torch.cat([self.resnet(x), self.densenet(x)], dim=1))

def set_phase(model, phase_b):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True
    if phase_b:
        for block in (model.resnet.layer4, model.densenet.features.denseblock4, model.densenet.features.norm5):
            for p in block.parameters():
                p.requires_grad = True

def set_train_mode(model, phase_b):
    model.train()
    model.resnet.eval()
    model.densenet.eval()
    if phase_b:
        model.resnet.layer4.train(True)
        model.densenet.features.denseblock4.train(True)
        model.densenet.features.norm5.train(True)
    model.head.train(True)

def build_optimizer(model, lr_head, lr_backbone=None):
    head_ids = {id(p) for p in model.head.parameters()}
    groups = [{'params': list(model.head.parameters()), 'lr': lr_head}]
    back = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    if back and lr_backbone:
        groups.append({'params': back, 'lr': lr_backbone})
    return optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def cosine_with_warmup(optimizer, total_epochs, warmup):
    base = optimizer.param_groups[0]['lr']
    def lr_lambda(epoch):
        if epoch < warmup:
            return (epoch + 1) / max(1, warmup)
        progress = (epoch - warmup) / max(1, total_epochs - warmup)
        return max(LR_MIN / base, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def run_phase(model, loader, criterion, n_epochs, lr_head, lr_backbone, phase_b):
    set_phase(model, phase_b)
    opt = build_optimizer(model, lr_head, lr_backbone)
    sched = cosine_with_warmup(opt, n_epochs, WARMUP_EPOCHS)
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    for _ in range(n_epochs):
        set_train_mode(model, phase_b)
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                loss = criterion(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
        sched.step()

def train_hybrid(idx, y, seed):
    set_seed(seed)
    pos = int(y.sum())
    neg = len(y) - pos
    cw = torch.tensor([len(y) / (2.0 * max(neg, 1)), len(y) / (2.0 * max(pos, 1))], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTHING)
    model = ResNetDenseNetHybrid(2).to(device)
    loader = make_loader(idx, y, augment=True, shuffle=True)
    run_phase(model, loader, criterion, PHASE_A_EPOCHS, LR_HEAD_A, None, phase_b=False)
    run_phase(model, loader, criterion, PHASE_B_EPOCHS, LR_HEAD_B, LR_BACKBONE_B, phase_b=True)
    model.eval()
    return model

def predict_slices(model, idx):
    loader = make_loader(idx, np.zeros(len(idx), dtype=int), augment=False, shuffle=False)
    out = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(x) + model(torch.flip(x, dims=[-1]))
            out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out)

previous = {}
for root, _, files in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'models' and 'results_memorization_xai' in root:
        for n in files:
            if n.endswith('.pt'):
                previous[n] = os.path.join(root, n)

def get_model(name, fit_idx, pos_cls, seed):
    path = os.path.join(MODEL_DIR, name + '.pt')
    if not os.path.exists(path) and name + '.pt' in previous:
        shutil.copy(previous[name + '.pt'], path)
    if os.path.exists(path):
        model = ResNetDenseNetHybrid(2).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        model.eval()
        print(name + ': reused')
        return model
    t0 = time.time()
    model = train_hybrid(fit_idx, np.isin(slice_label[fit_idx], pos_cls).astype(int), seed)
    torch.save(model.state_dict(), path)
    print(f'{name}: trained in {(time.time() - t0) / 60:.1f} min')
    return model

In [ ]:
def build_extractor(name):
    if name == 'resnet18+densenet121':
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        class Both(nn.Module):
            def __init__(self):
                super().__init__()
                self.r, self.d = r, d
            def forward(self, x):
                return torch.cat([self.r(x), self.d(x)], dim=1)
        return Both()
    if name == 'resnet50':
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None)
        m.fc = nn.Identity()
        return m
    if name == 'densenet121':
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Identity()
        return m
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    m.classifier = nn.Identity()
    return m

feats = {}
loader = make_loader(np.arange(n_slices), np.zeros(n_slices, dtype=int), augment=False, shuffle=False)
for name in BACKBONES:
    ext = build_extractor(name).to(device).eval()
    out = []
    with torch.no_grad():
        for x, _ in loader:
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                out.append(ext(x.to(device)).float().cpu().numpy())
    feats[name] = np.concatenate(out)
    del ext
    if device.type == 'cuda':
        torch.cuda.empty_cache()

def make_lr(n_fit):
    return make_pipeline(StandardScaler(), PCA(n_components=min(PCA_DIM, n_fit - 1), random_state=SEED),
                         LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
print('Features:', {k: v.shape[1] for k, v in feats.items()})

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 210MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 204MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 169MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 177MB/s]


Features: {'resnet18+densenet121': 1536, 'resnet50': 2048, 'densenet121': 1024, 'efficientnet_b0': 1280}


## Fold 1 of each split (repeat 0 of the subject split, as in the paper)

In [ ]:
y_pat = np.array([patient_label[p] for p in patients])
tr, te = next(StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=100).split(np.zeros(len(patients)), y_pat))
tr_p = [patients[i] for i in tr]
te_p = [patients[i] for i in te]
fit_p, cal_p = train_test_split(tr_p, test_size=CAL_FRACTION, stratify=[patient_label[p] for p in tr_p], random_state=1000)
subj = {'fit': idx_of(fit_p), 'test': idx_of(te_p)}

tr_s, te_s = next(StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=200).split(np.zeros(n_slices), slice_label))
fit_s, _ = train_test_split(tr_s, test_size=CAL_FRACTION, stratify=slice_label[tr_s], random_state=2000)
slc = {'fit': np.sort(fit_s), 'test': np.sort(te_s)}
print(f"subject split: {len(fit_p)} fitting and {len(te_p)} test subjects | slice split: {len(slc['fit'])} fitting slices, "
      f"{np.isin(slice_pid[slc['test']], slice_pid[slc['fit']]).mean() * 100:.0f}% of test slices from fitting subjects")

subject split: 204 fitting and 64 test subjects | slice split: 5447 fitting slices, 100% of test slices from fitting subjects


## Check 1: training fit under the subject split

In [ ]:
def macro_f1(y, p):
    y, p = np.asarray(y), np.asarray(p)
    f = []
    for c in range(3):
        tp = np.sum((y == c) & (p == c))
        fp = np.sum((y != c) & (p == c))
        fn = np.sum((y == c) & (p != c))
        f.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn > 0 else 0.0)
    return float(np.mean(f))

def mean_sd(v):
    v = np.asarray(v, dtype=float)
    return f'{v.mean():.3f} ± {v.std(ddof=1):.3f}' if len(v) > 1 else f'{v.mean():.3f}'

def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows():
        out.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
    return '\n'.join(out)

def subject_pred(prob3, idx):
    df = pd.DataFrame(prob3)
    df['pid'] = slice_pid[idx]
    g = df.groupby('pid').mean()
    return np.array([patient_label[p] for p in g.index]), g.values.argmax(1)

def fit_row(label, fit3, test3):
    ys, ps = subject_pred(fit3, subj['fit'])
    yt, pt = subject_pred(test3, subj['test'])
    return {'model': label,
            'fit slices F1': round(macro_f1(slice_label[subj['fit']], fit3.argmax(1)), 3),
            'fit subjects F1': round(macro_f1(ys, ps), 3),
            'test slices F1': round(macro_f1(slice_label[subj['test']], test3.argmax(1)), 3),
            'test subjects F1': round(macro_f1(yt, pt), 3)}

fit_prob, test_prob = [], []
for t, task in enumerate(OVA_TASKS):
    m = get_model('subject_' + task, subj['fit'], [t], SEED + t)
    fit_prob.append(predict_slices(m, subj['fit']))
    test_prob.append(predict_slices(m, subj['test']))
    if task == 'AD_vs_REST':
        subj_ad = m
    else:
        del m
        if device.type == 'cuda':
            torch.cuda.empty_cache()
rows = [fit_row('Hybrid, OvA (fine-tuned)', np.stack(fit_prob, 1), np.stack(test_prob, 1))]
for bb in BACKBONES:
    clf = make_lr(len(subj['fit']))
    clf.fit(feats[bb][subj['fit']], slice_label[subj['fit']])
    rows.append(fit_row(f'Frozen {bb} + LR', clf.predict_proba(feats[bb][subj['fit']]), clf.predict_proba(feats[bb][subj['test']])))
fit_df = pd.DataFrame(rows)
print(fit_df.to_string(index=False))

subject_AD_vs_REST: trained in 6.3 min
subject_CN_vs_REST: trained in 6.2 min
subject_MCI_vs_REST: trained in 6.2 min
                           model  fit slices F1  fit subjects F1  test slices F1  test subjects F1
        Hybrid, OvA (fine-tuned)          1.000            1.000           0.375             0.386
Frozen resnet18+densenet121 + LR          0.808            0.981           0.408             0.497
            Frozen resnet50 + LR          0.692            0.875           0.375             0.385
         Frozen densenet121 + LR          0.803            0.955           0.414             0.411
     Frozen efficientnet_b0 + LR          0.785            0.947           0.424             0.428


## Check 2: Grad-CAM of the AD-vs-rest network, slice split versus subject split, on the same slices

In [ ]:
slice_ad = get_model('slice_AD_vs_REST', slc['fit'], [AD], SEED)

common = np.intersect1d(subj['test'], slc['test'])
rng = np.random.default_rng(SEED)
cam_idx = np.sort(rng.choice(common, size=min(N_CAM, len(common)), replace=False))
RIM_PX = max(1, round(12 * IMAGE_SIZE / 224))
CORE_PX = max(1, round(40 * IMAGE_SIZE / 224))

def regions(i):
    m = IMAGES[i] > 10
    lab, n = ndimage.label(m)
    if n == 0:
        return None
    brain = ndimage.binary_fill_holes(lab == (np.argmax(ndimage.sum(m, lab, range(1, n + 1))) + 1))
    inner = ndimage.binary_erosion(brain, iterations=RIM_PX)
    core = ndimage.binary_erosion(brain, iterations=CORE_PX)
    if core.sum() == 0:
        return None
    return {'outside': ~brain, 'rim': brain & ~inner, 'core': core}

def gradcam(model, i):
    x = normalize(torch.from_numpy(IMAGES[i].copy()).unsqueeze(0).repeat(3, 1, 1).float() / 255.0).unsqueeze(0).to(device)
    x.requires_grad_(True)
    store = {}
    h1 = model.resnet.layer4.register_forward_hook(lambda mod, a, o: store.update(r=o))
    h2 = model.densenet.features.register_forward_hook(lambda mod, a, o: store.update(d=o))
    model.eval()
    with torch.enable_grad():
        out = model(x)
        score = out[0, 1] - out[0, 0]
        gr, gd = torch.autograd.grad(score, [store['r'], store['d']])
    h1.remove()
    h2.remove()
    cams = []
    for a, g in ((store['r'], gr), (store['d'], gd)):
        c = torch.relu((g.mean(dim=(2, 3), keepdim=True) * a.detach()).sum(1, keepdim=True))
        c = F.interpolate(c, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()
        if c.sum() > 1e-12:
            cams.append(c / c.sum())
    # a branch whose map is all zero (every gradient negative) carries no location and is left out
    return (sum(cams) / len(cams) if cams else None), float(torch.sigmoid(score).item())

rec, examples = [], {}
n_empty = 0
for i in cam_idx:
    reg = regions(i)
    if reg is None:
        continue
    row = {'pid': slice_pid[i], 'label': int(slice_label[i])}
    for name, rm in reg.items():
        row['area_' + name] = float(rm.mean())
    maps = {}
    for tag, m in (('subject', subj_ad), ('slice', slice_ad)):
        maps[tag], row[tag + '_p_AD'] = gradcam(m, i)
    if maps['subject'] is None or maps['slice'] is None:
        n_empty += 1
        continue
    for tag in ('subject', 'slice'):
        for name, rm in reg.items():
            row[tag + '_' + name] = float(maps[tag][rm].sum())
    if row['label'] not in examples:
        examples[row['label']] = (i, maps)
    rec.append(row)
cam_df = pd.DataFrame(rec)

def subject_boot(values, groups):
    g = pd.Series(values).groupby(groups).mean().values
    rng = np.random.default_rng(SEED)
    bs = [rng.choice(g, len(g)).mean() for _ in range(N_BOOT)]
    return float(g.mean()), [float(v) for v in np.percentile(bs, [2.5, 97.5])]

cam_rows = []
for name, label in (('outside', 'outside the brain'), ('rim', f'outer {RIM_PX}-pixel rim of the brain'), ('core', 'brain centre')):
    d, ci = subject_boot((cam_df['slice_' + name] - cam_df['subject_' + name]).values, cam_df['pid'].values)
    cam_rows.append({'region': label, 'share of image': round(cam_df['area_' + name].mean(), 3),
                     'subject-split network': round(cam_df['subject_' + name].mean(), 3),
                     'slice-split network': round(cam_df['slice_' + name].mean(), 3),
                     'difference (slice minus subject), 95% CI over subjects': f'{d:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}]'})
cam_tab = pd.DataFrame(cam_rows)
y_ad = cam_df['label'].values == AD
acc = {tag: float(np.mean((cam_df[tag + '_p_AD'].values >= 0.5) == y_ad)) for tag in ('subject', 'slice')}
print(f"{len(cam_df)} slices from {cam_df['pid'].nunique()} subjects ({n_empty} left out with an empty map) | "
      f"AD-vs-rest accuracy on these slices: subject split {acc['subject']:.3f}, slice split {acc['slice']:.3f}")
print(cam_tab.to_string(index=False))

fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
axes = np.atleast_2d(axes)
for r, lbl in enumerate(sorted(examples)):
    i, maps = examples[lbl]
    axes[r, 0].imshow(IMAGES[i], cmap='gray')
    axes[r, 0].set_title(f'{CLASS_NAMES[lbl]} slice')
    for c, tag in enumerate(('subject', 'slice'), start=1):
        axes[r, c].imshow(IMAGES[i], cmap='gray')
        axes[r, c].imshow(maps[tag], cmap='jet', alpha=0.45)
        axes[r, c].set_title(f'{tag}-split network')
for a in axes.ravel():
    a.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'xai_examples.png'), dpi=150)
plt.close(fig)

slice_AD_vs_REST: trained in 6.3 min
300 slices from 63 subjects (0 left out with an empty map) | AD-vs-rest accuracy on these slices: subject split 0.780, slice split 0.983
                         region  share of image  subject-split network  slice-split network difference (slice minus subject), 95% CI over subjects
              outside the brain           0.432                  0.434                0.407                                -0.039 [-0.074, -0.006]
outer 12-pixel rim of the brain           0.130                  0.140                0.131                                -0.009 [-0.017, -0.002]
                   brain centre           0.195                  0.175                0.201                                +0.033 [+0.009, +0.057]


## Save results

In [ ]:
lines = ['## Check 1: fit on own fitting subjects versus test subjects (subject split, fold 1 of repeat 0, macro-F1)', md_table(fit_df), '',
         f"## Check 2: Grad-CAM mass by region, {len(cam_df)} slices from {cam_df['pid'].nunique()} subjects that are test slices in both splits "
         f"({n_empty} left out because a network's map was empty)",
         f"AD-vs-rest slice accuracy on these slices: subject split {acc['subject']:.3f}, slice split {acc['slice']:.3f}", '',
         md_table(cam_tab), '']
with open(os.path.join(OUT_DIR, 'memorization_xai.md'), 'w') as fh:
    fh.write('\n'.join(lines))
with open(os.path.join(OUT_DIR, 'memorization_xai.json'), 'w') as fh:
    json.dump({'fit': fit_df.to_dict('records'), 'gradcam': cam_tab.to_dict('records'), 'accuracy_on_cam_slices': acc,
               'n_cam_slices': int(len(cam_df)), 'n_cam_subjects': int(cam_df['pid'].nunique()), 'n_empty_maps': n_empty,
               'rim_px': RIM_PX, 'core_px': CORE_PX}, fh, indent=1)
print('Saved to', OUT_DIR)

Saved to /kaggle/working/results_memorization_xai
